In [7]:
import chardet
import chardet, csv, pandas as pd, io, os, textwrap
from collections.abc import Mapping
from collections import OrderedDict
from collections import defaultdict, Counter
from datetime import datetime
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path
import re
import sys
from typing import Any, List


### CAUSALITY Project Notebook Mark Two - Disconnected

A notebook to ingest vuln data, add rating labels, and calculate stats on ratings and watchlist coverage. This version can run in a disconnected network assuming the data files are present in the HEAD directory. 

Usage: In a venv, install any missing modules in the cell above. Next define the input files below. The next cell will do a directory listing of the current path to help locate where the notebook is relative to the data files. Resale and /or incorporation into a paid product or service is not covered by license. See https://github.com/opendr-io/causality/blob/main/LICENSE.md for details. 

In [8]:
print("Current directory is:")
print(Path.cwd())
print("Here is the current directory listing - make sure your data files are here:")
for p in sorted(Path(".").iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
    tag = "<DIR>" if p.is_dir() else "     "
    print(f"{tag}  {p.name}")

Current directory is:
D:\causality-main\notebooks
Here is the current directory listing - make sure your data files are here:
<DIR>  .ipynb_checkpoints
       2025-ratings-sep-14.txt
       causality-disconnected.ipynb
       causality.ipynb
       causality_output_2026-03-17.csv
       causality_output_2026-03-19.csv
       causality_output_2026-03-20.csv
       counter.ipynb
       lines_with_wrong_comma_count.txt
       pdates.md
       reduction.txt
       times.csv
       times_new.csv
       times_scratch.md
       vdata.csv
       vuln-analysis.ipynb


### Configure these input file names: 

- vuln_path - your vuln data in csv format. Most of the export I see are in csv, let me know if you need a json ingestor. Ratings are loaded from the HEAD folder where the latest ratings live.


In [9]:
# Define your vuln data file name here - note these are file names not paths.
vuln_path = 'vdata.csv'  # Identify the vuln data file to ingest

if not Path(vuln_path).exists():
    raise FileNotFoundError(f"vuln_path not found: {Path(vuln_path).resolve()}\nUpdate vuln_path above to point to your data file.")

In [10]:
# Load all ratings files from the HEAD folder one level up
head_dir = Path('..') / 'HEAD'
rating_frames = {2024: [], 2025: [], 2026: []}

def sniff_sep(path, enc):
    """Return the delimiter by file extension or by sniffing the first line."""
    ext = path.suffix.lower()
    if ext == '.tsv':
        return '\t'
    if ext == '.csv':
        return ','
    with open(path, encoding=enc, errors='replace') as f:
        sample = f.readline()
    return '\t' if '\t' in sample else ','

all_files = sorted(
    list(head_dir.glob("*.txt")) +
    list(head_dir.glob("*.csv")) +
    list(head_dir.glob("*.tsv"))
)

for rating_file in all_files:
    with open(rating_file, "rb") as f:
        enc = chardet.detect(f.read())['encoding'] or 'utf-8'

    sep = sniff_sep(rating_file, enc)
    df = pd.read_csv(rating_file, sep=sep, low_memory=False, encoding=enc)
    df.columns = df.columns.str.strip()

    # Normalize column names
    if "cveID" in df.columns:
        df.rename(columns={"cveID": "cveid"}, inplace=True)
    if "Predicted_Label" in df.columns:
        df.rename(columns={"Predicted_Label": "rating"}, inplace=True)
    if "cwe_primary" in df.columns:
        df.rename(columns={"cwe_primary": "cwe_id"}, inplace=True)

    # Drop columns not in the standard schema
    drop_cols = [c for c in ('state', 'assigner') if c in df.columns]
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

    # Detect year from filename (e.g. 2024-output-may-24.txt -> 2024)
    year_match = re.search(r'\b(20\d{2})\b', rating_file.stem)
    if year_match:
        year = int(year_match.group(1))
    else:
        # Fall back: infer from most common CVE year in the data
        cve_col = next((c for c in ('cveid', 'cve') if c in df.columns), None)
        if cve_col:
            years = df[cve_col].str.extract(r'CVE-(\d{4})-')[0].dropna().astype(int)
            year = int(years.mode()[0]) if not years.empty else None
        else:
            year = None

    if year in rating_frames:
        rating_frames[year].append(df)
        print(f"✅ Loaded {rating_file.name} → {year}, shape {df.shape}")
    else:
        print(f"⚠️  Skipping {rating_file.name} (year {year} not handled)")

rating2024 = pd.concat(rating_frames[2024], ignore_index=True) if rating_frames[2024] else pd.DataFrame()
rating2025 = pd.concat(rating_frames[2025], ignore_index=True) if rating_frames[2025] else pd.DataFrame()
rating2026 = pd.concat(rating_frames[2026], ignore_index=True) if rating_frames[2026] else pd.DataFrame()

for name, df in [("rating2024", rating2024), ("rating2025", rating2025), ("rating2026", rating2026)]:
    if df.empty:
        print()
        print(f"⚠️ {name} is empty")
        continue
    print()
    print(f"Unique value counts for {name} (shape {df.shape}):")
    for col in df.columns:
        print(f'  {col}: {df[col].nunique(dropna=True)} unique values')
    all_nan = [col for col in df.columns if df[col].isna().all()]
    if all_nan:
        print(f"  ⚠️ Fields entirely NaN: {all_nan}")
    else:
        print("  ✅ No fields entirely NaN")

✅ Loaded 2024-output-may-24.txt → 2024, shape (37529, 5)
✅ Loaded 2025-processed-clean.txt → 2025, shape (41690, 16)
✅ Loaded 2026-MARCH-RUN.csv → 2026, shape (5943, 16)

Unique value counts for rating2024 (shape (37529, 5)):
  cveid: 37529 unique values
  vendorProject: 6584 unique values
  product: 11464 unique values
  shortDescription: 35784 unique values
  rating: 2 unique values
  ✅ No fields entirely NaN

Unique value counts for rating2025 (shape (41690, 16)):
  cveid: 41690 unique values
  published: 37718 unique values
  vulnerabilityname: 29237 unique values
  shortdescription: 40052 unique values
  vendorproject: 8990 unique values
  product: 18093 unique values
  vendor_has_multiple_values: 2 unique values
  product_has_multiple_values: 2 unique values
  cvss_score: 93 unique values
  cvss_severity: 5 unique values
  cvss_vector: 3069 unique values
  cwes: 1465 unique values
  kev: 2 unique values
  rating: 5 unique values
  cwe_id: 562 unique values
  cwe_name: 431 unique 

In [11]:
# rating2024 and rating2025 are loaded above from ../HEAD


In [12]:
path = vuln_path

# Detect encoding
with open(path, "rb") as f:
    raw = f.read()
det = chardet.detect(raw)
enc = det["encoding"] or "utf-8"

# Read header using csv.reader to count true fields (quotes respected)
first_newline_idx = raw.find(b"\n")
header_bytes = raw[: first_newline_idx if first_newline_idx != -1 else len(raw)]
header_text = header_bytes.decode(enc, errors="replace")
header_fields = next(csv.reader([header_text]))
expected_fields = len(header_fields)
expected_commas = expected_fields - 1

def count_commas_outside_quotes(line: str) -> int:
    """Counts commas that act as delimiters by scanning for double quotes.
    Handles doubled double-quotes "" as escaped quotes inside a quoted field."""
    count = 0
    i = 0
    in_quotes = False
    while i < len(line):
        ch = line[i]
        if ch == '"':
            if in_quotes:
                # If this is an escaped quote (""), skip the next one and stay in quotes
                if i + 1 < len(line) and line[i+1] == '"':
                    i += 2
                    continue
                else:
                    in_quotes = False
            else:
                in_quotes = True
        elif ch == ',' and not in_quotes:
            count += 1
        i += 1
    return count

offending_lines = []
with open(path, "r", encoding=enc, errors="replace", newline="") as f:
    for lineno, line in enumerate(f, start=1):
        if lineno == 1:
            continue
        commac = count_commas_outside_quotes(line.rstrip("\r\n"))
        if commac != expected_commas:
            offending_lines.append(lineno)

summary = {
    "Detected encoding": enc,
    "Fields in header": expected_fields,
    "Expected delimiter commas per line": expected_commas,
    "Total lines (including header)": raw.count(b"\n") + 1,
    "Offending line count (too few or too many commas)": len(offending_lines),
}

# Report results
if len(offending_lines) == 0:
    print("✅ No offending lines detected. File structure looks consistent.")
else:
    N = 25
    preview_df = pd.DataFrame({"offending_line_number": offending_lines[:N]})
    print("❌ First offending line numbers (up to 25)", preview_df)

    # Save full list for review
    full_list_path = "lines_with_wrong_comma_count.txt"
    with open(full_list_path, "w", encoding="utf-8") as out:
        out.write(f"# Header fields: {expected_fields} (expected commas per line: {expected_commas})\n")
        out.write("\n".join(map(str, offending_lines)))

    print(f"Full offending line list saved to {full_list_path}")

summary


✅ No offending lines detected. File structure looks consistent.


{'Detected encoding': 'utf-8',
 'Fields in header': 6,
 'Expected delimiter commas per line': 5,
 'Total lines (including header)': 3901,
 'Offending line count (too few or too many commas)': 0}

In [13]:
# Ingest data from the file in vuln_path defined in cell 3 above. 
# Detect encoding and handle errors (encoding may vary) and do some quality checks
with open(vuln_path, 'rb') as file:
    result = chardet.detect(file.read())
    encoding = result['encoding']
print(f"Detected encoding: {encoding}")

if len(offending_lines) > 0:
    sys.exit(
        f"❌ DATA ERRORS! See previous cell which found {len(offending_lines)} offending lines in {vuln_path}.\n"
        f"First few bad lines: {offending_lines[:10]}"
    )

# Load with error handling in case the detected encoding has issues
try:
    vulns = pd.read_csv(vuln_path, low_memory=False, encoding=encoding, header=0)
    print(f"✓ Loaded successfully with {encoding} encoding")
except UnicodeDecodeError:
    print(f"⚠️ Failed with {encoding}, trying with error handling...")
    vulns = pd.read_csv(
        vuln_path, low_memory=False, encoding=encoding,
        encoding_errors='replace', header=0
    )
    print("✓ Loaded with character replacement")

vulns.columns = vulns.columns.str.strip().str.lower()

all_nan = [col for col in vulns.columns if vulns[col].isna().all()]
if all_nan:
    print("\n⚠️ Fields entirely NaN:")
    for col in all_nan:
        print(f" - {col}")
else:
    print("\n✅ No fields are entirely NaN")

print("Shape of the vulns dataframe is:", vulns.shape)
print()
print("Unique value counts per field:\n")
for col in vulns.columns:
    n_unique = vulns[col].nunique(dropna=True)
    print(f"{col}: {n_unique} unique values")

print()
print("Fields with NaN values in vulns:\n")
nan_counts = vulns.isna().sum()
total_rows = len(vulns)

cols_with_nan = nan_counts[nan_counts > 0]
if len(cols_with_nan) > 0:
    for col, count in cols_with_nan.items():
        nan_pct = (count / total_rows) * 100
        print(f"{col}: {count} NaN values ({nan_pct:.2f}%)")
else:
    print("✅ No columns have NaN values")



Detected encoding: utf-8
✓ Loaded successfully with utf-8 encoding

✅ No fields are entirely NaN
Shape of the vulns dataframe is: (3899, 6)

Unique value counts per field:

cve: 3899 unique values
vendorproject: 1642 unique values
product: 2479 unique values
shortdescription: 3858 unique values
rating: 5 unique values
vulnerabilityname: 1545 unique values

Fields with NaN values in vulns:

vendorproject: 514 NaN values (13.18%)
product: 451 NaN values (11.57%)
vulnerabilityname: 2315 NaN values (59.37%)


In [14]:
# Try to find columns containing 'severity' or 'status'

severity_cols = [col for col in vulns.columns if 'severity' in col.lower()]
status_cols = [col for col in vulns.columns if 'status' in col.lower()]

# Print unique values for severity columns
if severity_cols:
    print("=== SEVERITY FIELDS ===\n")
    for col in severity_cols:
        unique_vals = vulns[col].dropna().unique()
        print(f"{col}:")
        print(f"  Unique values: {unique_vals}")
        print(f"  Count: {len(unique_vals)}\n")
else:
    print("No fields containing 'severity' found\n")

# Print unique values for status columns
if status_cols:
    print("=== STATUS FIELDS ===\n")
    for col in status_cols:
        unique_vals = vulns[col].dropna().unique()
        print(f"{col}:")
        print(f"  Unique values: {unique_vals}")
        print(f"  Count: {len(unique_vals)}\n")
else:
    print("No fields containing 'status' found")

# Find columns containing 'cve' or 'vulnerability' (case insensitive)
matching_cols = [col for col in vulns.columns 
                 if 'cve' in col.lower() or 'vulnerability' in col.lower()]

if matching_cols:
    print(f"Found {len(matching_cols)} field(s) containing 'cve' or 'vulnerability':\n")
    for col in matching_cols:
        print(f"  - {col}")
else:
    print("No fields containing 'cve' or 'vulnerability' were found")
    print(vulns.columns.tolist())



No fields containing 'severity' found

No fields containing 'status' found
Found 2 field(s) containing 'cve' or 'vulnerability':

  - cve
  - vulnerabilityname


In [15]:
# Check out the field list above and identify your field that contains CVE IDs. Provide it to the function below
# so that we have normalized field names across dataframes.

SOURCE_CVE_FIELD = 'cve'  # <-- change this as needed
colmap = {c.lower(): c for c in vulns.columns}
if SOURCE_CVE_FIELD.lower() in colmap:
    src = colmap[SOURCE_CVE_FIELD.lower()]
    if src == 'cve':
        pass  # already named 'cve'
    elif 'cve' in vulns.columns:
        print("Target column 'cve' already exists; skipping rename to avoid duplicate.")
    else:
        vulns.rename(columns={src: 'cve'}, inplace=True)
        print(f"Renamed '{SOURCE_CVE_FIELD}' to cve in order to play with the other dataframes.")
else:
    print(f"Column '{SOURCE_CVE_FIELD}' not found; nothing to rename. These could be CVE field names:")
    # Find columns containing 'cve' or 'vulnerability' (case insensitive)
    matching_cols = [col for col in vulns.columns 
                     if 'cve' in col.lower() or 'vulnerability' in col.lower()]
    
    if matching_cols:
        print(f"\nFound {len(matching_cols)} field(s) containing 'cve' or 'vulnerability':\n")
        for col in matching_cols:
            print(f"  - {col}")
    else:
        print("No fields containing 'cve' or 'vulnerability' were found")


In [16]:
# Regex for CVE IDs
cve_pattern = re.compile(r"^CVE-\d{4}-\d+$")
# Rows where 'cve' is invalid
bad_cve_rows = vulns[~vulns['cve'].astype(str).str.match(cve_pattern)]

if not bad_cve_rows.empty:
    print()
    print(f"❌ Found {len(bad_cve_rows)} rows with invalid CVE values!")
    # Print the full row for each offending record
    for idx, row in bad_cve_rows.iterrows():
        print(f"\n--- Row {idx} ---")
        print(row.to_string())
else:
    print("✅ All values in 'cve' column look like valid CVEs")
    print(vulns.columns.tolist())

✅ All values in 'cve' column look like valid CVEs
['cve', 'vendorproject', 'product', 'shortdescription', 'rating', 'vulnerabilityname']


In [17]:
# Re-extract year for ALL rows
vulns['year'] = vulns['cve'].astype(str).str.extract(r'CVE-(\d{4})-', expand=False)
vulns['year'] = pd.to_numeric(vulns['year'], errors='coerce').astype('Int64')

# Verify every row was processed
print(f"Total rows in vulns: {len(vulns):,}")
print(f"Rows with year value (not NaN): {vulns['year'].notna().sum():,}")
print(f"Rows with NaN year: {vulns['year'].isna().sum():,}")

# Print all unique values in the year field, including NaN
print("Unique values in 'year' field (including NaN):")
unique_years = vulns['year'].unique()
print(sorted([y for y in unique_years if pd.notna(y)]))

# Show if there are any NaN values
nan_count = vulns['year'].isna().sum()
if nan_count > 0:
    print(f"\nNaN values: {nan_count:,}")
    
# Set pandas to display all rows
pd.set_option('display.max_rows', None)

# Count all values in year field including NaN
print("Year value counts (including NaN):")
print(vulns['year'].value_counts(dropna=False).sort_index())


# Regex for a 4-digit year
year_pattern = re.compile(r"^\d{4}$")

# Make sure it's a string so regex can run safely
year_as_str = vulns['year'].astype(str)

# Rows where year is NaN, empty, or not matching 4-digit pattern
bad_year_rows = vulns[~year_as_str.str.match(year_pattern)]

if not bad_year_rows.empty:
    print(f"❌ Found {len(bad_year_rows)} rows without a valid year!")
    for idx, row in bad_year_rows.iterrows():
        print(f"\n--- Row {idx} ---")
        print(row.to_string())
else:
    print("✅ All rows have a valid year in the 'year' field")


# Reset display option if desired
# pd.reset_option('display.max_rows')

Total rows in vulns: 3,899
Rows with year value (not NaN): 3,899
Rows with NaN year: 0
Unique values in 'year' field (including NaN):
[np.int64(2024), np.int64(2025), np.int64(2026)]
Year value counts (including NaN):
year
2024    1876
2025    2022
2026       1
Name: count, dtype: Int64
✅ All rows have a valid year in the 'year' field


In [18]:
# Initialize the 'rating' field with 'cold'
vulns['rating'] = 'cold'
print("Initialized 'rating' field with 'cold'")

# Verify required columns exist
if 'year' not in vulns.columns:
    raise KeyError("vulns is missing 'year' column")
if SOURCE_CVE_FIELD not in vulns.columns:
    raise KeyError(f"vulns is missing '{SOURCE_CVE_FIELD}' column (SOURCE_CVE_FIELD)")

def apply_ratings(vulns, ratings_df, year, label):
    """Match ratings_df['cveid'] to vulns[SOURCE_CVE_FIELD] for a given year and apply ratings."""
    if 'cveid' not in ratings_df.columns or 'rating' not in ratings_df.columns:
        print(f'⚠️ {label} is missing cveid or rating column')
        return
    ratings_dict = dict(zip(
        ratings_df['cveid'].astype('string').str.strip().str.upper(),
        ratings_df['rating']
    ))
    mask = vulns['year'] == year
    total = mask.sum()
    if total == 0:
        print(f'{label}: No {year} CVEs found in vulns')
        return
    mapped = vulns.loc[mask, SOURCE_CVE_FIELD].astype('string').str.strip().str.upper().map(ratings_dict)
    matched = mapped.dropna()
    vulns.loc[matched.index, 'rating'] = matched
    print(f'{label}: {len(matched):,} matched out of {total:,} CVEs ({(len(matched)/total*100):.2f}% matched)')

apply_ratings(vulns, rating2025, 2025, '2025 ratings')
apply_ratings(vulns, rating2024, 2024, '2024 ratings')
apply_ratings(vulns, rating2026, 2026, '2026 ratings')

# Summary
print()
print('Rating distribution in vulns:')
print(vulns['rating'].value_counts())

# Group by year and rating
combo_counts = (
    vulns.groupby(['year', 'rating'], dropna=False)
         .size()
         .reset_index(name='count')
)
total = combo_counts['count'].sum()
combo_counts['percent'] = (combo_counts['count'] / total * 100).round(2)

# Check for missing ratings
bad_rating_rows = vulns[vulns['rating'].isna() | (vulns['rating'].astype(str).str.strip() == '')]
if not bad_rating_rows.empty:
    print(f'❌ Found {len(bad_rating_rows)} rows with no rating!')
    for idx, row in bad_rating_rows.iterrows():
        print(f'--- Row {idx} ---')
        print(row.to_string())
else:
    print()
    print('✅ All rows have a value in the rating field')
    print()
print(combo_counts)

Initialized 'rating' field with 'cold'
2025 ratings: 2,022 matched out of 2,022 CVEs (100.00% matched)
2024 ratings: 1,876 matched out of 1,876 CVEs (100.00% matched)
2026 ratings: 1 matched out of 1 CVEs (100.00% matched)

Rating distribution in vulns:
rating
cold       2515
hot         939
warm        305
fire        136
sunspot       4
Name: count, dtype: int64

✅ All rows have a value in the rating field

   year   rating  count  percent
0  2024     cold   1182    30.32
1  2024      hot    694    17.80
2  2025     cold   1333    34.19
3  2025     fire    136     3.49
4  2025      hot    245     6.28
5  2025  sunspot      3     0.08
6  2025     warm    305     7.82
7  2026  sunspot      1     0.03


In [19]:
# Filter for anything rated above cold
not_cold = vulns[vulns['rating'] != 'cold']

# Create filename with current date
current_date = datetime.now().strftime('%Y-%m-%d')
output_file = f'causality_output_{current_date}.csv'

# Export to CSV
not_cold.to_csv(output_file, index=False, encoding='utf-8')

print(f'Exported {len(not_cold):,} rows with non-cold ratings to {output_file!r}')
print('Breakdown:')
print(not_cold['rating'].value_counts())


Exported 1,384 rows with non-cold ratings to 'causality_output_2026-03-20.csv'
Breakdown:
rating
hot        939
warm       305
fire       136
sunspot      4
Name: count, dtype: int64


In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import math

# Build dropdown options from current vulns dataframe
year_opts = ['All'] + sorted(vulns['year'].dropna().unique().astype(int).tolist())
rating_opts = sorted(vulns['rating'].dropna().unique().tolist())

dd_year     = widgets.Dropdown(options=year_opts,            value="All", description="Year:")
dd_max_rows = widgets.Dropdown(options=[10, 100, 300, 1000], value=100,   description="Max rows:")

# One checkbox per rating
rating_checks = {
    r: widgets.Checkbox(value=True, description=r, indent=False,
                        layout=widgets.Layout(width='120px'))
    for r in rating_opts
}
rating_box = widgets.VBox(
    [widgets.Label("Rating:")] + [widgets.HBox(list(rating_checks.values()))],
    layout=widgets.Layout(border='1px solid #ddd', padding='6px', margin='0 6px')
)

out = widgets.Output()

# One checkbox per column, all checked by default
all_cols = list(vulns.columns)
col_checks = {col: widgets.Checkbox(value=True, description=col, indent=False,
                                    layout=widgets.Layout(width='220px'))
              for col in all_cols}

# Arrange checkboxes in rows of 4
cols_per_row = 4
check_rows = [
    widgets.HBox(list(col_checks.values())[i:i + cols_per_row])
    for i in range(0, len(col_checks), cols_per_row)
]
btn_all  = widgets.Button(description="Select all",  button_style='', layout=widgets.Layout(width='100px'))
btn_none = widgets.Button(description="Select none", button_style='', layout=widgets.Layout(width='100px'))

def on_all(b):
    for cb in col_checks.values():
        cb.value = True
def on_none(b):
    for cb in col_checks.values():
        cb.value = False

btn_all.on_click(on_all)
btn_none.on_click(on_none)

fields_box = widgets.VBox(
    [widgets.HBox([widgets.Label("Fields to show:"), btn_all, btn_none])] + check_rows,
    layout=widgets.Layout(border='1px solid #ddd', padding='6px', margin='6px 0')
)

def render_records(df, visible_cols, max_rows=100):
    """Render each row as a collapsible vertical field: value block."""
    if df.empty:
        return "<p><em>No matching rows.</em></p>"
    show = [c for c in visible_cols if c in df.columns]
    if not show:
        return "<p><em>No fields selected.</em></p>"
    blocks = []
    for idx, row in df.head(max_rows).iterrows():
        fields_html = "".join(
            f'<tr>'
            f'<td style="font-weight:bold;white-space:nowrap;padding:2px 12px 2px 4px;'
            f'vertical-align:top;color:#555;min-width:160px">{col}</td>'
            f'<td style="padding:2px 4px;word-break:break-word">{row[col]}</td>'
            f'</tr>'
            for col in show
        )
        blocks.append(
            f'<details open style="margin-bottom:6px;border:1px solid #ccc;'
            f'border-radius:4px;padding:2px 6px">'
            f'<summary style="cursor:pointer;font-weight:bold;padding:4px 2px">'
            f'Row {idx}</summary>'
            f'<table style="border-collapse:collapse;width:100%;font-size:0.92em">'
            f'{fields_html}</table>'
            f'</details>'
        )
    note = (f'<p style="color:#888"><em>Showing first {max_rows} of {len(df):,} rows.</em></p>'
            if len(df) > max_rows else '')
    return note + "".join(blocks)

def filter_vulns(change=None):
    df = vulns.copy()
    if dd_year.value != 'All':
        df = df[df['year'] == dd_year.value]
    selected_ratings = [r for r, cb in rating_checks.items() if cb.value]
    if selected_ratings and len(selected_ratings) < len(rating_opts):
        df = df[df['rating'].isin(selected_ratings)]
    visible = [col for col, cb in col_checks.items() if cb.value]
    with out:
        clear_output(wait=True)
        print(f'{len(df):,} matching rows')
        display(HTML(render_records(df, visible, max_rows=dd_max_rows.value)))

dd_year.observe(filter_vulns,     names="value")
dd_max_rows.observe(filter_vulns, names="value")
for cb in rating_checks.values():
    cb.observe(filter_vulns, names="value")
for cb in col_checks.values():
    cb.observe(filter_vulns, names="value")

display(widgets.HBox([dd_year, dd_max_rows]))
display(rating_box)
display(fields_box)
display(out)
filter_vulns()

Output()

The cell below is for optional troubleshooting; if you have a a csv file with some unusual chars in it they can be hard to find.

In [21]:
# For ingest troubleshooting
# - Detects encoding
# - Scans for problematic / non-ASCII characters (e.g., NBSP 0xA0, zero-width, smart quotes, dashes, BOM)
# - Reports counts and example line numbers for each issue
# - Flags lines with unbalanced double quotes (potential CSV breakage)
#
# You can tweak the `PROBLEM_CHARS` map to add/remove characters to check.
pd.set_option('display.max_rows', None)

# Read as bytes and detect encoding
with open(vuln_path, "rb") as f:
    raw = f.read()

det = chardet.detect(raw)
encoding = det.get("encoding") or "utf-8"
confidence = det.get("confidence")

# Decode with 'replace' so we can see if any undecodable bytes exist (as �)
text = raw.decode(encoding, errors="replace")

# Define characters/symbols to check
PROBLEM_CHARS = {
    "NBSP (U+00A0)": "\u00A0",
    "Zero Width Space (U+200B)": "\u200B",
    "Zero Width No-Break Space / BOM (U+FEFF)": "\uFEFF",
    "Left Double Smart Quote (U+201C)": "\u201C",
    "Right Double Smart Quote (U+201D)": "\u201D",
    "Left Single Smart Quote (U+2018)": "\u2018",
    "Right Single Smart Quote (U+2019)": "\u2019",
    "En Dash (U+2013)": "\u2013",
    "Em Dash (U+2014)": "\u2014",
    "Ellipsis (U+2026)": "\u2026",
    "Minus Sign (U+2212)": "\u2212",
    "Non-ASCII Thin Space (U+2009)": "\u2009",
    "Replacement Char from decoding (U+FFFD)": "\uFFFD",
}

# Split into lines preserving line numbers (1-based)
lines = text.splitlines()

# Helper: find all occurrences with line numbers
char_occurrences = {name: [] for name in PROBLEM_CHARS}
any_non_ascii_occurs = []
unbalanced_quote_lines = []

for i, line in enumerate(lines, start=1):
    # Check specific characters
    for name, ch in PROBLEM_CHARS.items():
        if ch in line:
            char_occurrences[name].append(i)
    # Track any non-ASCII characters (exclude \t and normal space/newline)
    for ch in line:
        if ord(ch) > 126 and ch not in ("\t",):
            any_non_ascii_occurs.append((i, ch))
    # CSV sanity: check unbalanced double quotes (") after accounting for CSV escaping "" -> treat them as pairs
    # Count number of raw quote characters; consider "" as two quotes (which is fine). We just need even total.
    if line.count('"') % 2 != 0:
        unbalanced_quote_lines.append(i)

# Summaries
problem_rows_summary = []
for name, lines_list in char_occurrences.items():
    count = len(lines_list)
    if count > 0:
        example_lines = ", ".join(map(str, lines_list[:10]))
    else:
        example_lines = ""
    problem_rows_summary.append({"Character": name, "Occurrences (lines)": count, "Example line numbers": example_lines})

# Non-ASCII summary (top characters)
non_ascii_counter = Counter(ch for _, ch in any_non_ascii_occurs)
top_non_ascii = [{"Char": k, "Codepoint": f"U+{ord(k):04X}", "Count": v} for k, v in non_ascii_counter.most_common(20)]

# Build DataFrames for display
df_chars = pd.DataFrame(problem_rows_summary).sort_values("Occurrences (lines)", ascending=False)
df_non_ascii = pd.DataFrame(top_non_ascii)
df_misc = pd.DataFrame(
    [
        {"Item": "Detected encoding", "Value": encoding},
        {"Item": "Detection confidence", "Value": confidence},
        {"Item": "Total lines", "Value": len(lines)},
        {"Item": "Lines with unbalanced quotes", "Value": len(unbalanced_quote_lines)},
    ]
)

# Also return a concise textual summary
summary_output = {
    "encoding": encoding,
    "confidence": confidence,
    "lines_total": len(lines),
    "unbalanced_quote_lines_count": len(unbalanced_quote_lines),
    "problem_char_types_with_hits": [row["Character"] for row in problem_rows_summary if row["Occurrences (lines)"] > 0],
}
summary_output
print()
print("Problem characters by type (line counts & examples)", df_chars)
print()
print("Top non-ASCII characters found (first 20 by frequency)", df_non_ascii)
print()
print("File / parsing sanity summary", df_misc)




Problem characters by type (line counts & examples)                                    Character  Occurrences (lines)  \
0                              NBSP (U+00A0)                  129   
7                           En Dash (U+2013)                   63   
6          Right Single Smart Quote (U+2019)                   62   
5           Left Single Smart Quote (U+2018)                   28   
8                           Em Dash (U+2014)                   10   
3           Left Double Smart Quote (U+201C)                    7   
4          Right Double Smart Quote (U+201D)                    7   
1                  Zero Width Space (U+200B)                    0   
2   Zero Width No-Break Space / BOM (U+FEFF)                    0   
9                          Ellipsis (U+2026)                    0   
10                       Minus Sign (U+2212)                    0   
11             Non-ASCII Thin Space (U+2009)                    0   
12   Replacement Char from decoding (U+FFFD)      